In [1]:
# Pyomo is an algebraic modeling language for Python. The `gdp` submodule
# adds Disjunction blocks for native generalized-disjunctive-programming
# modeling; a TransformationFactory step later rewrites them into a
# standard MILP that any LP/MIP solver can take.
import pyomo.environ as pyo
from pyomo.gdp import Disjunction

Sets

$\mathcal{I} = \{1, \ldots, N\}$ rectangles

Parameters

$w_i$ width of rectangle $i \in \mathcal{I}$ \
$\ell_i$ length of rectangle $i \in \mathcal{I}$ \
$W$ strip width (fixed)

Variables

$x_i \ge 0$ near corner along the length, $y_i \ge 0$ near corner across the width \
$L \ge 0$ strip length (objective)

Objective and Constraints

\begin{gather}
 \min_{x, y, L} \; L \\
 \textrm{s.t.} \quad x_i + \ell_i \le L \quad \forall i \in \mathcal{I} \\
 y_i + w_i \le W \quad \forall i \in \mathcal{I} \\
 x_i, y_i, L \ge 0
\end{gather}

Disjunction (no overlap)

For every pair $(i, j)$ with $i < j$, at least one of four geometric separations must hold &mdash; rectangle $i$ must lie to the left of, right of, below, or above rectangle $j$:

\begin{gather}
 [x_i + \ell_i \le x_j] \;\vee\; [x_j + \ell_j \le x_i]
 \;\vee\;
 \begin{bmatrix} y_i + w_i \le y_j \\ x_i + \ell_i \ge x_j + 1 \\ x_j + \ell_j \ge x_i + 1 \end{bmatrix}
 \;\vee\;
 \begin{bmatrix} y_j + w_j \le y_i \\ x_i + \ell_i \ge x_j + 1 \\ x_j + \ell_j \ge x_i + 1 \end{bmatrix}
\end{gather}

The two extra inequalities in the above/below disjuncts break *encoding degeneracy* (Trespalacios & Grossmann): in the plain four-disjunct model a pair separated both across the width and along the length satisfies two different disjuncts, so the same packing has multiple Boolean encodings and branch-and-bound explores each one. Requiring the above/below disjuncts to also overlap lengthwise routes every such arrangement uniquely through left/right; the $+1$ closes the edge-flush tie and is valid because all dimensions are integer.

Symmetry breaking

Every packing has mirror images with the same $L$ &mdash; reflect it across the strip's centerline or along its length. Pinning the center of the largest rectangle $r$ into the lower-left quadrant keeps one representative of each mirror class without cutting off the optimum:

\begin{gather}
 x_r + \tfrac{\ell_r}{2} \le \tfrac{L}{2}, \qquad y_r + \tfrac{w_r}{2} \le \tfrac{W}{2}
\end{gather}

Rectangles with identical $(w_i, \ell_i)$ are also interchangeable &mdash; swapping two identical pieces gives a "different" solution with the same $L$ &mdash; so each identical group is ordered by $x_i \le x_j$ (the reference rectangle's group is exempt, so the ordering cannot conflict with the quadrant pin).

In [2]:
# Problem instance. Eight rectangles with mixed shapes -- some tall, some
# wide, some square -- so the optimal packing is visually interesting.
# Strip width W = 10.
data = {
    "rects":  [1, 2, 3, 4, 5, 6, 7, 8],
    "w":      {1: 2.0, 2: 3.0, 3: 4.0, 4: 5.0, 5: 2.0, 6: 3.0, 7: 6.0, 8: 1.0},
    "length": {1: 6.0, 2: 4.0, 3: 2.0, 4: 3.0, 5: 5.0, 6: 3.0, 7: 2.0, 8: 7.0},
    "W":      10.0,
}

In [3]:
# Build the GDP model.
# ConcreteModel binds parameter values at construction time (as opposed to
# AbstractModel, which is parameterized and only instantiated later).
m = pyo.ConcreteModel()

rects = list(data["rects"])
W = float(data["W"])

# Loose upper bound on the strip length: stacking every rectangle end-to-end
# along the length is a (worst-case) feasible packing. Used as both an
# x-coordinate bound and an L bound so the gdp.bigm transformation can
# derive a sensible Big-M automatically from the variable bounds.
L_max = float(sum(data["length"][i] for i in rects))

# Index set over rectangles.
m.RECTS = pyo.Set(initialize=rects, ordered=True)

# Parameters: data the solver doesn't change.
m.w = pyo.Param(m.RECTS, initialize={i: data["w"][i] for i in rects})
m.length = pyo.Param(m.RECTS, initialize={i: data["length"][i] for i in rects})
m.W = pyo.Param(initialize=W)

# Decision variables. x runs along the strip length (bounded by L_max),
# y across the fixed width (bounded by W). Explicit bounds let the GDP
# transformation pick a tight Big-M for each disjunct.
m.x = pyo.Var(m.RECTS, domain=pyo.NonNegativeReals, bounds=(0.0, L_max))
m.y = pyo.Var(m.RECTS, domain=pyo.NonNegativeReals, bounds=(0.0, W))
m.L = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0.0, L_max))

# Objective: minimize strip length.
m.total_length = pyo.Objective(expr=m.L, sense=pyo.minimize)

# Containment: every rectangle fits inside the strip. fit_x ties the far
# end (along the length) to L; fit_y keeps it within the width W.
m.fit_x = pyo.Constraint(m.RECTS, rule=lambda m, i: m.x[i] + m.length[i] <= m.L)
m.fit_y = pyo.Constraint(m.RECTS, rule=lambda m, i: m.y[i] + m.w[i] <= m.W)

# Non-overlap disjunctions. For every unordered pair (i, j) with i < j,
# the rectangles must be separated in at least one of four ways. A
# `Disjunction` accepts a list of disjuncts, each being a list of
# constraint expressions; the GDP transformation later picks one and
# enforces it via Big-M / Hull / etc.
pairs = [(i, j) for idx, i in enumerate(rects) for j in rects[idx + 1:]]
m.PAIRS = pyo.Set(initialize=pairs, dimen=2)

# The above/below disjuncts carry two extra inequalities -- the
# degeneracy-breaking form of Trespalacios & Grossmann. In the plain
# model the four regions overlap: a pair separated both across the width
# and along the length can be encoded by two different disjunct
# selections, and branch-and-bound explores both encodings of the same
# packing. Requiring above/below to also overlap lengthwise by >= 1
# routes every such arrangement uniquely through left/right; the +1
# closes the edge-flush tie and is valid because all dimensions are
# integer.
def disj_rule(m, i, j):
    return [
        [m.x[i] + m.length[i] <= m.x[j]],       # i left of j
        [m.x[j] + m.length[j] <= m.x[i]],       # i right of j
        [m.y[i] + m.w[i] <= m.y[j],             # i below j ...
         m.x[i] + m.length[i] >= m.x[j] + 1,    # ... and lengthwise
         m.x[j] + m.length[j] >= m.x[i] + 1],   #     overlap >= 1
        [m.y[j] + m.w[j] <= m.y[i],             # i above j ...
         m.x[i] + m.length[i] >= m.x[j] + 1,
         m.x[j] + m.length[j] >= m.x[i] + 1],
    ]
m.no_overlap = Disjunction(m.PAIRS, rule=disj_rule)

# Symmetry breaking: every packing has mirror images (reflect it across
# the strip's centerline or along its length) with the same L, and
# branch-and-bound would prove the same bound separately for each one.
# Pinning the largest rectangle's CENTER to the lower-left quadrant keeps
# one representative per mirror class without cutting off the optimum.
ref = max(rects, key=lambda i: data["w"][i] * data["length"][i])
m.sym_x = pyo.Constraint(expr=m.x[ref] + m.length[ref] / 2.0 <= m.L / 2.0)
m.sym_y = pyo.Constraint(expr=m.y[ref] + m.w[ref] / 2.0 <= m.W / 2.0)

# Identical rectangles are interchangeable; order each identical group
# along the length (x) to keep one representative per permutation. The
# reference rectangle's group is exempt so the ordering cannot conflict
# with the quadrant pin.
groups = {}
for i in rects:
    groups.setdefault((data["w"][i], data["length"][i]), []).append(i)
m.lex = pyo.ConstraintList()
for members in groups.values():
    if len(members) < 2 or ref in members:
        continue
    for a, b in zip(members, members[1:]):
        m.lex.add(m.x[a] <= m.x[b])

In [4]:
# Reformulate the GDP into a standard MILP. Big-M is the classical choice --
# fewest variables, often the loosest LP relaxation. Other transformations
# available in pyomo.gdp:
#   gdp.mbigm -- per-constraint tight Big-M (midpoint of bigm and hull)
#   gdp.hull  -- disaggregated variables, tighter LP relaxation
pyo.TransformationFactory("gdp.bigm").apply_to(m)

# Solve with HiGHS (ships as a pip wheel via highspy).
# tee=True streams the solver log to stdout so we can see HiGHS's branch-
# and-bound progress.
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(m, tee=True)

# Optimal strip length and per-rectangle near-corner coordinates.
print(f"\nOptimal strip length L = {pyo.value(m.L):.3f}")
print(f"\n{'rect':>4}  {'x':>6}  {'y':>6}  {'w':>4}  {'length':>6}")
for i in rects:
    print(
        f"{i:>4}  "
        f"{pyo.value(m.x[i]):>6.2f}  "
        f"{pyo.value(m.y[i]):>6.2f}  "
        f"{data['w'][i]:>4.1f}  "
        f"{data['length'][i]:>6.1f}"
    )

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms


MIP has 270 rows; 129 cols; 811 nonzeros; 112 integer variables (112 binary)
Coefficient ranges:
  Matrix  [5e-01, 4e+01]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 3e+01]
  RHS     [1e+00, 3e+01]
Presolving model
246 rows, 124 cols, 752 nonzeros 0s
245 rows, 123 cols, 750 nonzeros 0s
Presolve reductions: rows 245(-25); columns 123(-6); nonzeros 750(-61) 

Solving MIP model with:
   245 rows
   123 cols (106 binary, 0 integer, 0 implied int., 17 continuous, 0 domain fixed)
   750 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  

 L       0       0         0   0.00%   7               10                30.00%     2396    104     95       460     1.0s

11.3% inactive integer columns, restarting
Model after restart has 201 rows, 106 cols (89 bin., 0 int., 0 impl., 17 cont., 0 dom.fix.), and 612 nonzeros

         0       0         0   0.00%   7               10                30.00%       45      0      0      2674     1.0s
         0       0         0   0.00%   7               10                30.00%       45     25      3      2712     1.0s
 L       0       0         0   0.00%   7               9.999999          30.00%     1951    107      3      3060     1.2s


 T    1021       0       381  99.95%   9               9                  0.00%     2317     69   2346     20088     2.9s
      1023       0       382 100.00%   9               9                  0.00%     2317     69   2346     20115     2.9s

Solving report
  Status            Optimal
  Primal bound      9
  Dual bound        9
  Gap               0% (tolerance: 0.01%)
  P-D integral      0.453799709096
  Solution status   feasible
                    9 (objective)
                    0 (bound viol.)
                    4.4408920985e-16 (int. viol.)
                    0 (row viol.)
  Timing            2.86
  Max sub-MIP depth 5
  Nodes             1023
  Repair LPs        0
  LP iterations     20115
                    3118 (strong br.)
                    2454 (separation)
                    3761 (heuristics)



Optimal strip length L = 9.000

rect       x       y     w  length
   1    0.00    0.00   2.0     6.0
   2    5.00    3.00   3.0     4.0
   3    7.00    6.00   4.0     2.0
   4    2.00    2.00   5.0     3.0
   5    2.00    7.00   2.0     5.0
   6    6.00   -0.00   3.0     3.0
   7    0.00    3.00   6.0     2.0
   8   -0.00    9.00   1.0     7.0
